In [2]:
# Install required libraries
!pip install requests beautifulsoup4 pandas sentence-transformers -q

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_properties():
    properties = []

    # Simulating real estate data scraping from Zillow-style structure
    sample_data = [
        {"title": "Modern Apartment in New Cairo", "price": "2,500,000 EGP", "location": "New Cairo, Cairo", "bedrooms": 3, "bathrooms": 2, "area_sqm": 150, "type": "Apartment"},
        {"title": "Villa in Sheikh Zayed", "price": "8,500,000 EGP", "location": "Sheikh Zayed, Giza", "bedrooms": 5, "bathrooms": 4, "area_sqm": 400, "type": "Villa"},
        {"title": "Studio in Maadi", "price": "950,000 EGP", "location": "Maadi, Cairo", "bedrooms": 1, "bathrooms": 1, "area_sqm": 60, "type": "Studio"},
        {"title": "Penthouse in Zamalek", "price": "12,000,000 EGP", "location": "Zamalek, Cairo", "bedrooms": 4, "bathrooms": 3, "area_sqm": 280, "type": "Penthouse"},
        {"title": "Apartment in 6th of October", "price": "1,800,000 EGP", "location": "6th of October, Giza", "bedrooms": 2, "bathrooms": 1, "area_sqm": 110, "type": "Apartment"},
        {"title": "Twin House in Madinaty", "price": "5,200,000 EGP", "location": "Madinaty, Cairo", "bedrooms": 4, "bathrooms": 3, "area_sqm": 320, "type": "Twin House"},
        {"title": "Chalet in North Coast", "price": "3,100,000 EGP", "location": "North Coast, Alexandria", "bedrooms": 3, "bathrooms": 2, "area_sqm": 180, "type": "Chalet"},
        {"title": "Office Space in Downtown", "price": "4,400,000 EGP", "location": "Downtown, Cairo", "bedrooms": 0, "bathrooms": 2, "area_sqm": 200, "type": "Commercial"},
        {"title": "Apartment in Heliopolis", "price": "2,100,000 EGP", "location": "Heliopolis, Cairo", "bedrooms": 3, "bathrooms": 2, "area_sqm": 140, "type": "Apartment"},
        {"title": "Villa in New Capital", "price": "9,800,000 EGP", "location": "New Capital, Cairo", "bedrooms": 6, "bathrooms": 5, "area_sqm": 500, "type": "Villa"},
    ]

    for item in sample_data:
        properties.append(item)
        time.sleep(0.1)  # Simulate request delay

    print(f"✅ Scraped {len(properties)} properties successfully")
    return properties

raw_data = scrape_properties()

✅ Scraped 10 properties successfully


In [4]:
# Data Cleaning & Structuring
def clean_and_structure(raw_data):
    df = pd.DataFrame(raw_data)

    # Clean price column - remove text, convert to number
    df['price_egp'] = df['price'].str.replace(' EGP', '').str.replace(',', '').astype(float)

    # Calculate price per sqm
    df['price_per_sqm'] = (df['price_egp'] / df['area_sqm']).round(2)

    # Extract city from location
    df['city'] = df['location'].apply(lambda x: x.split(',')[-1].strip())

    # Add property tier based on price
    def get_tier(price):
        if price < 2000000:
            return 'Affordable'
        elif price < 5000000:
            return 'Mid-Range'
        elif price < 10000000:
            return 'Premium'
        else:
            return 'Luxury'

    df['market_tier'] = df['price_egp'].apply(get_tier)

    # Drop original price column
    df.drop(columns=['price'], inplace=True)

    print("✅ Data cleaned and structured successfully")
    print(f"\n📊 Summary:")
    print(f"Total Properties: {len(df)}")
    print(f"Property Types: {df['type'].unique()}")
    print(f"Market Tiers: {df['market_tier'].value_counts().to_dict()}")
    print(f"\n🏠 Sample Output:")
    print(df[['title', 'price_egp', 'price_per_sqm', 'market_tier', 'city']].to_string())

    return df

cleaned_df = clean_and_structure(raw_data)

✅ Data cleaned and structured successfully

📊 Summary:
Total Properties: 10
Property Types: ['Apartment' 'Villa' 'Studio' 'Penthouse' 'Twin House' 'Chalet'
 'Commercial']
Market Tiers: {'Mid-Range': 4, 'Premium': 3, 'Affordable': 2, 'Luxury': 1}

🏠 Sample Output:
                           title   price_egp  price_per_sqm market_tier        city
0  Modern Apartment in New Cairo   2500000.0       16666.67   Mid-Range       Cairo
1          Villa in Sheikh Zayed   8500000.0       21250.00     Premium        Giza
2                Studio in Maadi    950000.0       15833.33  Affordable       Cairo
3           Penthouse in Zamalek  12000000.0       42857.14      Luxury       Cairo
4    Apartment in 6th of October   1800000.0       16363.64  Affordable        Giza
5         Twin House in Madinaty   5200000.0       16250.00     Premium       Cairo
6          Chalet in North Coast   3100000.0       17222.22   Mid-Range  Alexandria
7       Office Space in Downtown   4400000.0       22000.00   Mi

In [5]:
# Semantic Structuring using Sentence Transformers
from sentence_transformers import SentenceTransformer
import numpy as np

def add_semantic_structure(df):
    print("⏳ Loading semantic model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Create rich text description for each property
    df['semantic_description'] = df.apply(lambda row:
        f"{row['type']} in {row['location']} | {row['bedrooms']} bed {row['bathrooms']} bath | "
        f"{row['area_sqm']}sqm | {row['market_tier']} segment | "
        f"Price per sqm: {row['price_per_sqm']} EGP", axis=1
    )

    # Generate embeddings
    print("⏳ Generating semantic embeddings...")
    embeddings = model.encode(df['semantic_description'].tolist())
    df['embedding_vector'] = embeddings.tolist()

    print(f"✅ Semantic embeddings generated for {len(df)} properties")
    print(f"📐 Embedding dimensions: {len(embeddings[0])}")
    print(f"\n📝 Sample semantic descriptions:")
    for desc in df['semantic_description'].head(3):
        print(f"  → {desc}")

    return df

structured_df = add_semantic_structure(cleaned_df)

⏳ Loading semantic model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Generating semantic embeddings...
✅ Semantic embeddings generated for 10 properties
📐 Embedding dimensions: 384

📝 Sample semantic descriptions:
  → Apartment in New Cairo, Cairo | 3 bed 2 bath | 150sqm | Mid-Range segment | Price per sqm: 16666.67 EGP
  → Villa in Sheikh Zayed, Giza | 5 bed 4 bath | 400sqm | Premium segment | Price per sqm: 21250.0 EGP
  → Studio in Maadi, Cairo | 1 bed 1 bath | 60sqm | Affordable segment | Price per sqm: 15833.33 EGP


In [6]:
# Save final structured data to CSV
structured_df.drop(columns=['embedding_vector']).to_csv('real_estate_structured.csv', index=False)
print("✅ Data saved to real_estate_structured.csv")

# Show final dataframe
print("\n📊 Final Structured Dataset:")
structured_df.drop(columns=['embedding_vector'])

✅ Data saved to real_estate_structured.csv

📊 Final Structured Dataset:


,title,location,bedrooms,bathrooms,area_sqm,type,price_egp,price_per_sqm,city,market_tier,semantic_description
0,Modern Apartment in New Cairo,"New Cairo, Cairo",3,2,150,Apartment,2500000.0,16666.67,Cairo,Mid-Range,"Apartment in New Cairo, Cairo | 3 bed 2 bath |..."
1,Villa in Sheikh Zayed,"Sheikh Zayed, Giza",5,4,400,Villa,8500000.0,21250.00,Giza,Premium,"Villa in Sheikh Zayed, Giza | 5 bed 4 bath | 4..."
2,Studio in Maadi,"Maadi, Cairo",1,1,60,Studio,950000.0,15833.33,Cairo,Affordable,"Studio in Maadi, Cairo | 1 bed 1 bath | 60sqm ..."
3,Penthouse in Zamalek,"Zamalek, Cairo",4,3,280,Penthouse,12000000.0,42857.14,Cairo,Luxury,"Penthouse in Zamalek, Cairo | 4 bed 3 bath | 2..."
4,Apartment in 6th of October,"6th of October, Giza",2,1,110,Apartment,1800000.0,16363.64,Giza,Affordable,"Apartment in 6th of October, Giza | 2 bed 1 ba..."
5,Twin House in Madinaty,"Madinaty, Cairo",4,3,320,Twin House,5200000.0,16250.00,Cairo,Premium,"Twin House in Madinaty, Cairo | 4 bed 3 bath |..."
6,Chalet in North Coast,"North Coast, Alexandria",3,2,180,Chalet,3100000.0,17222.22,Alexandria,Mid-Range,"Chalet in North Coast, Alexandria | 3 bed 2 ba..."
7,Office Space in Downtown,"Downtown, Cairo",0,2,200,Commercial,4400000.0,22000.00,Cairo,Mid-Range,"Commercial in Downtown, Cairo | 0 bed 2 bath |..."
8,Apartment in Heliopolis,"Heliopolis, Cairo",3,2,140,Apartment,2100000.0,15000.00,Cairo,Mid-Range,"Apartment in Heliopolis, Cairo | 3 bed 2 bath ..."
9,Villa in New Capital,"New Capital, Cairo",6,5,500,Villa,9800000.0,19600.00,Cairo,Premium,"Villa in New Capital, Cairo | 6 bed 5 bath | 5..."
